# ICMS 150 (2 Day experiment summary)

- Closed loop plan: day 1: perform visual (16 orientations, 100 trials) and electrical (3000 patterns, 50 oracles x 20 trials). This notebook is used after output spike sorting and curation to fit stimulus-encoder TCN model and generate stimualtion patterns for each orientation to try on day 2. 


Visual stim duration: 0.5s ON, 0.5s OFF
Velocity(temporal frequency): 3Hz
Width (spatial frequency): 0.05 cycle/degree
After that is 4000 patterns, 3000 samples appear 1 times, 50 oracle samples repeating 20 times


In [1]:
import sys
sys.path.insert(0, '../patterns_5k')

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from utils import read_pattern_json, preprocess_pattern_stimulations_df
%load_ext autoreload
%autoreload 2
day1_datadir = "data/icms_150_6_2_26"
day2_datadir = "data/icms_150_6_3_26"
mode = "pca"

all_spikes_day1 = np.load(f"{day1_datadir}/All_Shank_Spk_Vecs.npy", allow_pickle=False) # contains each spike indexed with (sample index, unit index, segment index)
all_spikes_day2 = np.load(f"{day2_datadir}/All_Shank_Spk_Vecs.npy", allow_pickle=False) # contains each spike indexed with (sample index, unit index, segment index)

spikes_df_day1 = pd.DataFrame(all_spikes_day1)  # columns auto-named from dtype fields
spikes_df_day1.rename(columns={'sample_index': 'timestamp', 'unit_index': 'neuron_id'}, inplace=True)
spikes_df_day1.drop(columns=['segment_index'], inplace=True)

spikes_df_day2 = pd.DataFrame(all_spikes_day2)  # columns auto-named from dtype fields
spikes_df_day2.rename(columns={'sample_index': 'timestamp', 'unit_index': 'neuron_id'}, inplace=True)
spikes_df_day2.drop(columns=['segment_index'], inplace=True)

pattern_df_day1, min_pattern_timestamp_day1 = preprocess_pattern_stimulations_df(read_pattern_json(f"{day1_datadir}/Combined_Pattern_Registrations.pkl"), align_to_stim=True)
pattern_df_day2, min_pattern_timestamp_day2 = preprocess_pattern_stimulations_df(read_pattern_json(f"{day2_datadir}/Combined_Pattern_Registrations.pkl"), align_to_stim=True)

visual_stim_times_day1= np.load(f"{day1_datadir}/VStim_Onset_TS.npy") * 30 # 1600
visual_stim_times_day2= np.load(f"{day2_datadir}/VStim_Onset_TS.npy") * 30 # 1600
visual_stim_labels_day1 = np.load(f"{day1_datadir}/VStim_Labels.npy") # 1600 
visual_stim_labels_day2 = np.load(f"{day2_datadir}/VStim_Labels.npy") # 1600

visual_orientation_degs1 = np.sort(np.unique(visual_stim_labels_day1))
visual_orientation_degs2 = np.sort(np.unique(visual_stim_labels_day2))
visual_trial_df1 = pd.DataFrame({
    'timestamp': visual_stim_times_day1,
    'orientation': visual_stim_labels_day1
})
visual_trial_df2 = pd.DataFrame({
    'timestamp': visual_stim_times_day2,
    'orientation': visual_stim_labels_day2
})
min_visual_timestamp_1 = visual_trial_df1['timestamp'].min()
max_visual_timestamp_1 = visual_trial_df1['timestamp'].max()
min_visual_timestamp_2 = visual_trial_df2['timestamp'].min()
max_visual_timestamp_2 = visual_trial_df2['timestamp'].max()

visual_spikes_1 = spikes_df_day1[(spikes_df_day1['timestamp'] >= min_visual_timestamp_1) & (spikes_df_day1['timestamp'] <= max_visual_timestamp_1)].copy()
visual_spikes_2 = spikes_df_day2[(spikes_df_day2['timestamp'] >= min_visual_timestamp_2) & (spikes_df_day2['timestamp'] <= max_visual_timestamp_2)].copy()

## Visual Tuning over 2 Days 

In [3]:

# --- Timing constants -------------------------------------------------
# Spike timestamps are in *samples* at SAMPLE_RATE Hz. The visual stimulus
# was shown for STIM_MS ms, so we bin the full stim window (no data is
# thrown away). The window is chopped into bins of BIN_MS ms.
SAMPLE_RATE    = 30_000                       # Hz
STIM_MS        = 500                          # stimulus duration (ms)
BIN_MS         = 10                           # PSTH bin width (ms)

SAMPLES_PER_MS = SAMPLE_RATE // 1000          # 30 samples / ms
WINDOW_LEN     = STIM_MS * SAMPLES_PER_MS     # 500 ms -> 15000 samples
BIN_SAMPLES    = BIN_MS  * SAMPLES_PER_MS     # 50 ms  -> 1500 samples
N_BINS         = WINDOW_LEN // BIN_SAMPLES    # STIM_MS / BIN_MS
assert WINDOW_LEN % BIN_SAMPLES == 0, "BIN_MS must evenly divide STIM_MS"

N_VIS_TRIALS = 100

def make_sum_responses(visual_spikes, visual_trial_df, visual_orientation_degs):
    all_neuron_ids = np.sort(visual_spikes['neuron_id'].unique())
    n_neurons = len(all_neuron_ids)
    neuron_to_idx = {n: i for i, n in enumerate(all_neuron_ids)}

    sum_responses = []
    trial_labels = []
    for orientation in visual_orientation_degs:
        sum_response = np.zeros((n_neurons, N_BINS))
        for trial in range(N_VIS_TRIALS):
            trial_time = visual_trial_df[visual_trial_df['orientation'] == orientation].iloc[trial]['timestamp']
            trial_spikes = visual_spikes[(visual_spikes['timestamp'] >= trial_time) &
                                        (visual_spikes['timestamp'] <  trial_time + WINDOW_LEN)]
            if trial_spikes.empty:
                continue
            bin_edges = trial_time + np.arange(N_BINS + 1) * BIN_SAMPLES  # N_BINS+1 edges across STIM_MS

            for neuron_id in trial_spikes['neuron_id'].unique():
                idx = neuron_to_idx[neuron_id]
                counts, _ = np.histogram(
                    trial_spikes[trial_spikes['neuron_id'] == neuron_id]['timestamp'],
                    bins=bin_edges,
                )
                sum_response[idx] += counts
        sum_response /= N_VIS_TRIALS  # average over trials 
        sum_responses.append(sum_response) 
    sum_responses = np.array(sum_responses)
    return sum_responses  # (n_orientations, n_neurons, N_BINS)
sum_responses_1 = make_sum_responses(visual_spikes_1, visual_trial_df1, visual_orientation_degs1)
sum_responses_2 = make_sum_responses(visual_spikes_2, visual_trial_df2, visual_orientation_degs2)
print(f"Window: {STIM_MS} ms = {WINDOW_LEN} samples at {SAMPLE_RATE} Hz")
print(f"Bins:   {N_BINS} x {BIN_MS} ms ({BIN_SAMPLES} samples each)")
print(f"sum_responses1 shape (orient, neuron, time_bin): {sum_responses_1.shape}")
print(f"sum_responses2 shape (orient, neuron, time_bin): {sum_responses_2.shape}")


Window: 500 ms = 15000 samples at 30000 Hz
Bins:   50 x 10 ms (300 samples each)
sum_responses1 shape (orient, neuron, time_bin): (16, 63, 50)
sum_responses2 shape (orient, neuron, time_bin): (16, 55, 50)


In [ ]:
import plotly.graph_objects as go
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA

# Compute 3-axis projection: [CIS, PC1, PC2]
def make_ring(sum_responses, visual_orientation_degs, visual_trial_df, visual_spikes, mode="pca", proj=None):
    n_neurons = sum_responses.shape[1]
    all_neuron_ids = np.sort(visual_spikes['neuron_id'].unique())
    neuron_to_idx = {n: i for i, n in enumerate(all_neuron_ids)}
    X = sum_responses.sum(axis=-1)             # (n_orientations, n_neurons)  -- per-orientation averages
    mean_vec = X.mean(axis=0, keepdims=True)


    # 1. Extract raw trial counts and labels to fit LDA
    vis_window_samples = (600 * SAMPLE_RATE) // 1000
    trial_raw_counts = []
    trial_labels = []

    for orientation in visual_orientation_degs:
        trial_times = visual_trial_df[visual_trial_df['orientation'] == orientation]['timestamp'].values[:N_VIS_TRIALS]
        for t_time in trial_times:
            trial_spikes = visual_spikes[(visual_spikes['timestamp'] >= t_time) &
                                        (visual_spikes['timestamp'] <  t_time + vis_window_samples)]
            x_trial = np.zeros(n_neurons)
            if not trial_spikes.empty:
                counts = trial_spikes.groupby('neuron_id').size()
                for nid, c in counts.items():
                    if nid in neuron_to_idx:
                        x_trial[neuron_to_idx[nid]] = c
            trial_raw_counts.append(x_trial)
            trial_labels.append(orientation)

    trial_raw_counts = np.array(trial_raw_counts)
    trial_labels = np.array(trial_labels)

    # 2. Fit LDA to visual trials
    # LDA requires categorical labels, so we map orientations to strings
    lda_labels = trial_labels.astype(str)

    # (Using n_components=2 to match PC plot)
    lda = LDA(n_components=2)
    lda.fit(trial_raw_counts, lda_labels)
    if proj is None: 
        if mode == "pca":
            pca3 = PCA(n_components=2)
            pca3.fit(X)
            proj, _ = np.linalg.qr(np.array([
                mean_vec[0],
                pca3.components_[0],
                pca3.components_[1],
            ]).T)        
            print ("PCA explained variance ratio:", pca3.explained_variance_ratio_)
        elif mode == "lda":
            proj, _ = np.linalg.qr(np.array([
                mean_vec[0],
                lda.scalings_[:, 0],
                lda.scalings_[:, 1],
            ]).T)
            print ("LDA explained variance ratio:", lda.explained_variance_ratio_)
        else:
            raise ValueError(f"Unknown mode: {mode}")
    

    coords = X @ proj                           # (n_orientations, 3)  -- averaged points

    # ---- Per-trial points: spike counts per (orientation, trial) summed over time bins ----
    trial_coords = []     # (n_orient * n_trials, 3)
    trial_doubled = []    # color matches its orientation
    for o_idx, orientation in enumerate(visual_orientation_degs):
        trial_times = visual_trial_df[visual_trial_df['orientation'] == orientation]['timestamp'].values[:N_VIS_TRIALS]
        for t_time in trial_times:
            trial_spikes = visual_spikes[(visual_spikes['timestamp'] >= t_time) &
                                        (visual_spikes['timestamp'] <  t_time + WINDOW_LEN)]
            x_trial = np.zeros(n_neurons)
            if not trial_spikes.empty:
                counts = trial_spikes.groupby('neuron_id').size()
                for nid, c in counts.items():
                    if nid in neuron_to_idx:
                        x_trial[neuron_to_idx[nid]] = c
            trial_coords.append(x_trial @ proj)
            trial_doubled.append((int(orientation) * 2) % 360)
    trial_coords = np.array(trial_coords)
    trial_doubled = np.array(trial_doubled)

    doubled = (np.array(visual_orientation_degs) * 2) % 360

    fig = go.Figure()

    # Per-trial scatter (low alpha)
    fig.add_trace(go.Scatter3d(
        x=trial_coords[:, 0], y=trial_coords[:, 1], z=trial_coords[:, 2],
        mode='markers',
        marker=dict(size=3, color=trial_doubled, colorscale='HSV',
                    cmin=0, cmax=360, opacity=0.05,
                    line=dict(width=0)),
        name='Individual trials',
        hovertemplate='trial<extra></extra>',
        showlegend=True,
    ))

    # Orientation-average scatter (full opacity, on top)
    fig.add_trace(go.Scatter3d(
        x=coords[:, 0], y=coords[:, 1], z=coords[:, 2],
        mode='markers+text',
        marker=dict(size=8, color=doubled, colorscale='HSV',
                    cmin=0, cmax=360,
                    colorbar=dict(
                        title='Orientation (°)',
                        tickvals=np.arange(0, 361, 72),
                        ticktext=[f'{int(v/2)}°' for v in np.arange(0, 361, 72)],
                    ),
                    line=dict(color='black', width=1)),
        text=[f'{int(o)}°' for o in visual_orientation_degs],
        textposition='top center',
        name='Orientation average',
    ))
    if mode == "pca":
        fig.update_layout(title='Average Neural Responses to Visual Stimuli (PCA Projection)')
        fig.update_layout(
        title='Average Neural Responses to Visual Stimuli',
        scene=dict(xaxis_title='CIS', yaxis_title='PC1', zaxis_title='PC2',
                aspectmode='data'),
        width=800, height=700,
    )
    elif mode == "lda":
        fig.update_layout(title='Average Neural Responses to Visual Stimuli (LDA Projection)')
        fig.update_layout(
        title='Average Neural Responses to Visual Stimuli',
        scene=dict(xaxis_title='CIS', yaxis_title='LDA1', zaxis_title='LDA2',
                aspectmode='data'),
        width=800, height=700,  
    )
    fig.show()
    return proj
day1_proj = make_ring(sum_responses_1, visual_orientation_degs1, visual_trial_df1, visual_spikes_1, mode="pca")


In [ ]:
day2_proj = make_ring(sum_responses_2, visual_orientation_degs2, visual_trial_df2, visual_spikes_2, mode="pca", proj=day1_proj)


## Stim Responses, Day 2 Only

In [2]:
target_mapping = pd.read_csv('data/icms_150_4_4_26/merged_pattern_xref.csv')
target_mapping

,pattern_name,source_file,source_budget,source_pattern_name,ori_idx,target_orientation_deg,start_timing_index,n_trials,steps_str
0,502,generated_pattern_registration_4budget_600ms.pkl,4,502,2,45.0,1,1,[0]ch57:dm2;ch74:dm2;ch88:dm-1;ch113:dm-1 | [1...
1,5011,generated_pattern_registration_4budget_600ms.pkl,4,5011,11,247.5,2,1,[0]ch57:dm2;ch74:dm2;ch88:dm-1;ch113:dm-1 | [1...
2,503,generated_pattern_registration_4budget_600ms.pkl,4,503,3,67.5,3,1,[0]ch57:dm2;ch74:dm2;ch88:dm-1;ch113:dm-1 | [1...
3,5010,generated_pattern_registration_4budget_600ms.pkl,4,5010,10,225.0,4,1,[0]ch57:dm2;ch74:dm2;ch88:dm-1;ch113:dm-1 | [1...
4,500,generated_pattern_registration_4budget_600ms.pkl,4,500,0,0.0,5,1,[0]ch57:dm2;ch74:dm2;ch88:dm-1;ch113:dm-1 | [1...
5,504,generated_pattern_registration_4budget_600ms.pkl,4,504,4,90.0,6,1,[0]ch57:dm2;ch74:dm2;ch91:dm-1;ch113:dm2 | [1]...
6,507,generated_pattern_registration_4budget_600ms.pkl,4,507,7,157.5,7,1,[0]ch57:dm2;ch74:dm2;ch88:dm-1;ch113:dm-1 | [1...
7,505,generated_pattern_registration_4budget_600ms.pkl,4,505,5,112.5,8,1,[0]ch57:dm2;ch78:dm0;ch113:dm2;ch127:dm-1 | [1...
8,5014,generated_pattern_registration_4budget_600ms.pkl,4,5014,14,315.0,9,1,[0]ch57:dm2;ch74:dm2;ch88:dm-1;ch113:dm-1 | [1...
9,5012,generated_pattern_registration_4budget_600ms.pkl,4,5012,12,270.0,10,1,[0]ch57:dm2;ch74:dm2;ch91:dm-1;ch113:dm2 | [1]...


In [ ]:
# Project observed averages, per-trial dots, and CNN predictions into PC1/PC2,
# computed from the same VIS_ORIENT_TIME window used to define the targets.
import os
import plotly.graph_objects as go
from sklearn.decomposition import PCA

VIS_ORIENT_TIME = 600  # ms window for visual orientation decoding (from above analysis)

if VIS_ORIENT_TIME > STIM_MS: 
    fac = VIS_ORIENT_TIME / STIM_MS
else:
    fac = 1
targets = sum_responses_2[:, :, :VIS_ORIENT_TIME // 10].sum(axis=2) * fac

               # (n_orient, n_neurons)
pca_win = PCA(n_components=2).fit(X_win)
proj_win = pca_win.components_.T                # (n_neurons, 2)
coords_win = X_win @ proj_win                   # (n_orient, 2)

# CNN predictions under greedy stims (already over the VIS_ORIENT_TIME window).
greedy_pred_full = np.zeros((len(visual_orientation_degs), n_neurons), dtype=np.float32)
greedy_coords_win =  @ proj_win

# Per-trial dots over the same VIS_ORIENT_TIME window.
vis_window_samples = (VIS_ORIENT_TIME * SAMPLE_RATE) // 1000
trial_coords  = []
trial_doubled = []
for o_idx, orientation in enumerate(visual_orientation_degs):
    trial_times = visual_trial_df[visual_trial_df['orientation'] == orientation]['timestamp'].values[:N_VIS_TRIALS]
    for t_time in trial_times:
        trial_spikes = visual_spikes[(visual_spikes['timestamp'] >= t_time) &
                                     (visual_spikes['timestamp'] <  t_time + vis_window_samples)]
        x_trial = np.zeros(n_neurons)
        if not trial_spikes.empty:
            counts = trial_spikes.groupby('neuron_id').size()
            for nid, c in counts.items():
                if nid in neuron_to_idx:
                    x_trial[neuron_to_idx[nid]] = c
        trial_coords.append(x_trial @ proj_win)
        trial_doubled.append((int(orientation) * 2) % 360)
trial_coords  = np.array(trial_coords)
trial_doubled = np.array(trial_doubled)

doubled = (np.array(visual_orientation_degs) * 2) % 360

fig = go.Figure()

# Per-trial cloud
fig.add_trace(go.Scatter(
    x=trial_coords[:, 0], y=trial_coords[:, 1],
    mode='markers',
    marker=dict(size=5, color=trial_doubled, colorscale='HSV',
                cmin=0, cmax=360, opacity=0.1, line=dict(width=0)),
    hoverinfo='skip', showlegend=False,
))

# Match links between observed and predicted
for o in range(len(visual_orientation_degs)):
    fig.add_trace(go.Scatter(
        x=[coords_win[o, 0], greedy_coords_win[o, 0]],
        y=[coords_win[o, 1], greedy_coords_win[o, 1]],
        mode='lines', line=dict(color='gray', width=1, dash='dot'),
        hoverinfo='skip', showlegend=False,
    ))

# Observed orientation averages (circles)
fig.add_trace(go.Scatter(
    x=coords_win[:, 0], y=coords_win[:, 1],
    mode='markers',
    marker=dict(size=10, color=doubled, colorscale='HSV', cmin=0, cmax=360,
                line=dict(color='black', width=1),
                colorbar=dict(
                    title='Orientation (°)',
                    tickvals=np.arange(0, 361, 72),
                    ticktext=[f'{int(v/2)}°' for v in np.arange(0, 361, 72)],
                )),
    showlegend=False,
))

# Greedy CNN predictions (diamonds)
fig.add_trace(go.Scatter(
    x=greedy_coords_win[:, 0], y=greedy_coords_win[:, 1],
    mode='markers',
    marker=dict(size=10, color=doubled, colorscale='HSV', cmin=0, cmax=360,
                symbol='diamond', line=dict(color='white', width=1)),
    showlegend=False,
))

fig.update_layout(
    title=f'Observed vs CNN-predicted responses ({VIS_ORIENT_TIME} ms window)',
    xaxis_title='PC1', yaxis_title='PC2',
    width=850, height=700,
)

os.makedirs('results', exist_ok=True)
fig.write_html(f'results/vis_response_pc1_pc2_recon_{BUDGET}.html', include_plotlyjs='cdn')
try:
    fig.write_image(f'results/vis_response_pc1_pc2_recon_{BUDGET}.svg', width=850, height=700, scale=2)
    fig.write_image(f'results/vis_response_pc1_pc2_recon_{BUDGET}.pdf', width=850, height=700, scale=2)
except Exception as e:
    print(f'[image export skipped: {e}]')
    print('Open the HTML in a browser and File > Print > Save as PDF for vector output.')
fig.show()
